In [ ]:


from __future__ import annotations
from dataclasses import dataclass
from typing import Any, Dict, List, Optional
import logging
import time
import uuid
import requests


# Basic logger setup. In production this would go to a log pipeline.
logger = logging.getLogger("mcp.salesforce")
logger.setLevel(logging.INFO)


@dataclass(frozen=True)
class UserContext:
    """Auth context for the current request.
    Contains identity + delegated Salesforce access.
    """
    user_id: str
    email: str
    groups: List[str]
    sf_user_id: str
    sf_instance_url: str
    sf_access_token: str


class Forbidden(Exception):
    """Raised when the user is not allowed to run the tool."""

class BackendError(Exception):
    """Raised when Salesforce fails or returns an unexpected error."""

class BadInput(Exception):
    """Raised when the caller passes invalid parameters."""

def _assert_allowed(ctx: UserContext) -> None:
    """Simple RBAC gate for this tool.
    Keeps logic explicit and easy to reason about.
    """
    if not any(g in ctx.groups for g in ("sales", "cs", "support")):
        raise Forbidden("user is not permitted to query opportunities")


def _soql_escape(value: str) -> str:
    """Escape minimal characters for SOQL string literals."""
    return value.replace("\\", "\\\\").replace("'", "\\'")


def _salesforce_query(ctx: UserContext, soql: str, timeout_s: float = 8.0) -> Dict[str, Any]:
    """Runs a SOQL query using Salesforce REST API."""
    url = f"{ctx.sf_instance_url}/services/data/v59.0/query"
    headers = {"Authorization": f"Bearer {ctx.sf_access_token}"}

    try:
        resp = requests.get(url, headers=headers, params={"q": soql}, timeout=timeout_s)
    except requests.Timeout as e:
        raise BackendError("salesforce timeout") from e
    except requests.RequestException as e:
        raise BackendError("salesforce request failed") from e

    if resp.status_code == 401:
        # Token expired or revoked.
        raise Forbidden("salesforce token rejected")
    if resp.status_code >= 400:
        # Keep error message short so logs don't leak data.
        raise BackendError(f"salesforce error status={resp.status_code}")

    return resp.json()


def get_my_open_opportunities(
    ctx: UserContext,
    *,
    min_amount: float,
    close_within_days: int = 30,
    limit: int = 25,
) -> Dict[str, Any]:
    """Return the caller's open opportunities over a threshold.
    Always scoped to the current user (OwnerId forced server-side).
    """
    req_id = str(uuid.uuid4())
    start = time.time()

    # ---- Input validation (failure scenario #1: bad request) ----
    if min_amount < 0 or min_amount > 10_000_000:
        raise BadInput("min_amount out of bounds")
    if close_within_days < 0 or close_within_days > 365:
        raise BadInput("close_within_days out of bounds")
    if limit < 1 or limit > 200:
        raise BadInput("limit out of bounds")

    logger.info("tool_start req_id=%s user=%s tool=get_my_open_opportunities", req_id, ctx.email)

    try:
        # ---- Authorization check (failure scenario #2: forbidden) ----
        _assert_allowed(ctx)

        # Force "my data" scoping. The caller cannot override this.
        owner_id = _soql_escape(ctx.sf_user_id)

        # Keep query allowlisted: no raw SOQL from the client.
        soql = f"""
        SELECT Id, Name, Amount, StageName, CloseDate, NextStep, Account.Id, Account.Name
        FROM Opportunity
        WHERE IsClosed = false
          AND OwnerId = '{owner_id}'
          AND Amount >= {min_amount}
          AND CloseDate = NEXT_N_DAYS:{close_within_days}
        ORDER BY Amount DESC
        LIMIT {limit}
        """.strip()

        raw = _salesforce_query(ctx, soql)

        # Allowlist output fields so we don't leak extra Salesforce columns.
        opportunities: List[Dict[str, Any]] = []
        for r in raw.get("records", []):
            acct = r.get("Account") or {}
            opportunities.append(
                {
                    "id": r.get("Id"),
                    "name": r.get("Name"),
                    "amount": r.get("Amount"),
                    "stage": r.get("StageName"),
                    "close_date": r.get("CloseDate"),
                    "next_step": r.get("NextStep"),
                    "account": {"id": acct.get("Id"), "name": acct.get("Name")},
                }
            )

        latency_ms = int((time.time() - start) * 1000)
        logger.info(
            "tool_success req_id=%s user=%s count=%d latency_ms=%d",
            req_id,
            ctx.email,
            len(opportunities),
            latency_ms,
        )

        return {
            "opportunities": opportunities,
            "meta": {"record_count": len(opportunities), "request_id": req_id},
        }

    except (Forbidden, BadInput) as e:
        # Known safe failures: log and re-raise.
        logger.warning("tool_denied req_id=%s user=%s err=%s", req_id, ctx.email, str(e))
        raise

    except BackendError as e:
        # Backend failure: log a safe message (no payload).
        logger.error("tool_backend_error req_id=%s user=%s err=%s", req_id, ctx.email, str(e))
        raise

    except Exception as e:
        # Catch-all: you don't want unhandled exceptions leaking details.
        logger.exception("tool_unexpected req_id=%s user=%s", req_id, ctx.email)
        raise BackendError("unexpected failure") from e
